In [1]:
import pandas as pd
import torch
import numpy as np
import matplotlib.pyplot as plt
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from gensim.utils import simple_preprocess
from nltk.tokenize import word_tokenize
import re
import logging
from gensim.matutils import cossim
from scipy.spatial.distance import cosine
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity
import nltk
nltk.download('punkt')
import statsmodels.api as sm
import ast


#pour voir temps d'avancement de l'entrainement
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)
#pour l'enlever
#logging.getLogger("gensim").setLevel(logging.ERROR)

/Users/mathieu/anaconda3/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
[nltk_data] Downloading package punkt to /Users/mathieu/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [2]:
chemin_fichier = "FOMC_statements_characteristics_050525.csv"
df_speaker=pd.read_csv(chemin_fichier)

In [3]:
df_speaker = df_speaker[df_speaker['Member']=='M']
df_speaker=df_speaker[['Date','Speaker','Statement']]

#colonne contenant la longueur du statement
df_speaker['Statement_length'] = df_speaker['Statement'].str.len()

#pour chaque individu on garde seulement sa déclaration la plus longue
df_speaker = df_speaker.loc[df_speaker.groupby(['Date', 'Speaker'])['Statement_length'].idxmax()].reset_index(drop=True)

In [4]:
tokenizer = SentenceTransformer('all-MiniLM-L6-v2')._first_module().tokenizer
df_speaker['token_count'] = df_speaker['Statement'].apply(lambda x: len(tokenizer.tokenize(x)))

2025-07-29 14:53:44,355 : INFO : Load pretrained SentenceTransformer: all-MiniLM-L6-v2
c:\Users\Mathieu\anaconda3\envs\sbert_env\lib\site-packages\huggingface_hub\file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
c:\Users\Mathieu\anaconda3\envs\sbert_env\lib\site-packages\huggingface_hub\file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
2025-07-29 14:53:47,564 : INFO : Use pytorch device_name: cpu
Token indices sequence length is longer than the specified maximum sequence length for this model (789 > 512). Running this sequence through the model will result in indexing errors


In [5]:
df_speaker

,Date,Speaker,Statement,Statement_length,token_count
0,1976-03-29,MR. BALLES,"Mr. Chairman, may I respond to Governor Coldwe...",1415.0,311
1,1976-03-29,MR. BLACK,You also have more constraint on your federal ...,90.0,17
2,1976-03-29,MR. BURNS,Very well. I want to make a comment on the Mem...,3740.0,789
3,1976-03-29,MR. COLDWELL,"If I could, Mr. Chairman, I’d like us to retur...",807.0,172
4,1976-03-29,MR. EASTBURN,"Yes, a better relationship between nonborrowed...",533.0,111
...,...,...,...,...,...
6472,2018-12-19,MS. BRAINARD,"Thank you. Like many of you, I see a disconnec...",8715.0,1686
6473,2018-12-19,MS. DALY,Let me begin by just saying that California ha...,8719.0,1817
6474,2018-12-19,MS. GEORGE,All right. I’m going to drop any literary refe...,4819.0,916
6475,2018-12-19,MS. LOGAN,"Thank you, Simon. Starting on your third exhib...",9703.0,1907


In [6]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)  # supprime les espaces multiples
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if re.match(r'\w+', t)]  # conserve seulement les mots (enlève ., ,, etc.)
    return tokens

In [7]:
df_filtered = df_speaker[['Date', 'Speaker', 'Statement', 'token_count']]
df_filtered

,Date,Speaker,Statement,token_count
0,1976-03-29,MR. BALLES,"Mr. Chairman, may I respond to Governor Coldwe...",311
1,1976-03-29,MR. BLACK,You also have more constraint on your federal ...,17
2,1976-03-29,MR. BURNS,Very well. I want to make a comment on the Mem...,789
3,1976-03-29,MR. COLDWELL,"If I could, Mr. Chairman, I’d like us to retur...",172
4,1976-03-29,MR. EASTBURN,"Yes, a better relationship between nonborrowed...",111
...,...,...,...,...
6472,2018-12-19,MS. BRAINARD,"Thank you. Like many of you, I see a disconnec...",1686
6473,2018-12-19,MS. DALY,Let me begin by just saying that California ha...,1817
6474,2018-12-19,MS. GEORGE,All right. I’m going to drop any literary refe...,916
6475,2018-12-19,MS. LOGAN,"Thank you, Simon. Starting on your third exhib...",1907


In [8]:
df_filtered['token_count'].mean()

924.5298749421029

In [9]:
df_filtered['token_count'].max()

4888

In [10]:
#entrainement du modele pour les déclarations individuelles
documents = [
    TaggedDocument(words=preprocess(row['Statement']), tags=[str(i)])
    for i, row in df_filtered.iterrows()
]

model = Doc2Vec(
    vector_size=300,        # taille standard du vecteur de sortie (peut monter à 400 si nécessaire)
    window=10,              # contexte large = bon pour documents longs
    min_count=5,            # ignore les mots rares (moins de 5 occurrences globales)
    workers=4,              
    epochs=100,              # un bon nombre pour convergence sur ce corpus
    dm=1,                   # 1 = Distributed Memory (meilleure pour classification en général)
    negative=10,            
    hs=0,                  
    sample=1e-5,            
)

model.build_vocab(documents)
model.train(documents, total_examples=model.corpus_count, epochs=model.epochs)

2025-07-29 14:56:42,295 : INFO : Doc2Vec lifecycle event {'params': 'Doc2Vec<dm/m,d300,n10,w10,mc5,s1e-05,t4>', 'datetime': '2025-07-29T14:56:42.295831', 'gensim': '4.3.3', 'python': '3.10.18 | packaged by Anaconda, Inc. | (main, Jun  5 2025, 13:08:55) [MSC v.1929 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.19045-SP0', 'event': 'created'}
2025-07-29 14:56:42,296 : INFO : collecting all words and their counts
2025-07-29 14:56:42,297 : INFO : PROGRESS: at example #0, processed 0 words (0 words/s), 0 word types, 0 tags
2025-07-29 14:56:43,126 : INFO : collected 42135 word types and 6477 unique tags from a corpus of 6477 examples and 5048236 words
2025-07-29 14:56:43,127 : INFO : Creating a fresh vocabulary
2025-07-29 14:56:43,191 : INFO : Doc2Vec lifecycle event {'msg': 'effective_min_count=5 retains 13518 unique words (32.08% of original 42135, drops 28617)', 'datetime': '2025-07-29T14:56:43.191542', 'gensim': '4.3.3', 'python': '3.10.18 | packaged by Anaconda, Inc. | (main, Jun  5 20

In [11]:
tags_used = [doc.tags[0] for doc in documents]  # exactement ce que le modèle a vu

df_speaker=df_filtered
df_speaker['embedding'] = [model.dv[tag] for tag in tags_used]

df_speaker=df_speaker[['Date','Speaker','Statement','embedding']]

In [12]:
def cosine_sim(vec1, vec2):
    return cosine_similarity([vec1], [vec2])[0][0]

def compute_influence_from_df(df_speaker):
    import numpy as np

    df = df_speaker.copy()
    df["influence"] = np.nan
    df["fs"] = np.nan
    df["bs"] = np.nan

    # Récupérer les dates dans l'ordre chronologique
    unique_dates = sorted(df["Date"].unique())

    # S'assurer qu'il y a au moins 3 dates pour pouvoir calculer quelque chose
    if len(unique_dates) < 3:
        raise ValueError("Il doit y avoir au moins 3 dates distinctes dans df_speaker")

    # Parcourir toutes les dates sauf la première et la dernière
    for i in range(1, len(unique_dates) - 1):
        date_before = unique_dates[i - 1]
        date_middle = unique_dates[i]
        date_after = unique_dates[i + 1]

        df_before = df[df["Date"] == date_before]
        df_middle = df[df["Date"] == date_middle]
        df_after = df[df["Date"] == date_after]

        emb_before = list(df_before["embedding"])
        emb_after = list(df_after["embedding"])

        for idx, row in df_middle.iterrows():
            vec = row["embedding"]

            fs = np.mean([cosine_sim(vec, ea) for ea in emb_after])
            bs = np.mean([cosine_sim(vec, eb) for eb in emb_before])

            influence = fs / bs if bs != 0 else float("inf")

            df.at[idx, "fs"] = fs
            df.at[idx, "bs"] = bs
            df.at[idx, "influence"] = influence

    return df

In [13]:
df_speaker

,Date,Speaker,Statement,embedding
0,1976-03-29,MR. BALLES,"Mr. Chairman, may I respond to Governor Coldwe...","[-0.46235126, 0.62222743, -0.33321473, -0.5694..."
1,1976-03-29,MR. BLACK,You also have more constraint on your federal ...,"[-0.35095495, -0.15901202, -0.14085643, 0.0324..."
2,1976-03-29,MR. BURNS,Very well. I want to make a comment on the Mem...,"[-1.7908938, -1.4453974, -0.23234199, 0.970562..."
3,1976-03-29,MR. COLDWELL,"If I could, Mr. Chairman, I’d like us to retur...","[-0.8694455, -1.1118317, -0.66208005, -0.24708..."
4,1976-03-29,MR. EASTBURN,"Yes, a better relationship between nonborrowed...","[-0.07844626, -0.10638859, -0.20478983, -0.322..."
...,...,...,...,...
6472,2018-12-19,MS. BRAINARD,"Thank you. Like many of you, I see a disconnec...","[0.3713909, 0.67823684, 0.24893859, -0.0692386..."
6473,2018-12-19,MS. DALY,Let me begin by just saying that California ha...,"[-1.5910221, 0.2616942, -0.7327649, -0.4738449..."
6474,2018-12-19,MS. GEORGE,All right. I’m going to drop any literary refe...,"[0.007266722, 0.33701637, -0.8229123, 0.419581..."
6475,2018-12-19,MS. LOGAN,"Thank you, Simon. Starting on your third exhib...","[0.55851716, 0.53137076, 3.3962665, 0.95060444..."


In [14]:
df_compare=compute_influence_from_df(df_speaker)

In [15]:
df_compare

,Date,Speaker,Statement,embedding,influence,fs,bs
0,1976-03-29,MR. BALLES,"Mr. Chairman, may I respond to Governor Coldwe...","[-0.46235126, 0.62222743, -0.33321473, -0.5694...",NaN,NaN,NaN
1,1976-03-29,MR. BLACK,You also have more constraint on your federal ...,"[-0.35095495, -0.15901202, -0.14085643, 0.0324...",NaN,NaN,NaN
2,1976-03-29,MR. BURNS,Very well. I want to make a comment on the Mem...,"[-1.7908938, -1.4453974, -0.23234199, 0.970562...",NaN,NaN,NaN
3,1976-03-29,MR. COLDWELL,"If I could, Mr. Chairman, I’d like us to retur...","[-0.8694455, -1.1118317, -0.66208005, -0.24708...",NaN,NaN,NaN
4,1976-03-29,MR. EASTBURN,"Yes, a better relationship between nonborrowed...","[-0.07844626, -0.10638859, -0.20478983, -0.322...",NaN,NaN,NaN
...,...,...,...,...,...,...,...
6472,2018-12-19,MS. BRAINARD,"Thank you. Like many of you, I see a disconnec...","[0.3713909, 0.67823684, 0.24893859, -0.0692386...",NaN,NaN,NaN
6473,2018-12-19,MS. DALY,Let me begin by just saying that California ha...,"[-1.5910221, 0.2616942, -0.7327649, -0.4738449...",NaN,NaN,NaN
6474,2018-12-19,MS. GEORGE,All right. I’m going to drop any literary refe...,"[0.007266722, 0.33701637, -0.8229123, 0.419581...",NaN,NaN,NaN
6475,2018-12-19,MS. LOGAN,"Thank you, Simon. Starting on your third exhib...","[0.55851716, 0.53137076, 3.3962665, 0.95060444...",NaN,NaN,NaN


In [ ]:
import json
#df_compare['embedding'] = df_compare['embedding'].apply(lambda x: json.dumps(x.tolist() if isinstance(x, np.ndarray) else x))
#df_compare.to_csv("dfdoc2vec2.csv", index=False)